# Sesión 1 (parte 2): Arquitectura de Datos en Google Cloud Platform

## Módulo 2: Arquitectura Actual de Datos en GCP

### Objetivos de este notebook
1. Explorar la arquitectura Medallion creada en el Módulo 1 usando `INFORMATION_SCHEMA`
2. Configurar Cloud Storage como capa de Data Lake complementaria a BigQuery
3. Crear tablas externas con **BigLake** sobre datos en GCS
4. Auditar jobs, costes y almacenamiento con vistas del sistema
5. Gestionar permisos IAM a nivel dataset y tabla
6. Configurar **Analytics Hub** para compartición segura de datos
7. Implementar patrones de separación dev/test/prod

**Prerrequisito**: Haber ejecutado el notebook del Módulo 1 (datasets `bronze_personio`, `silver_personio`, `gold_people_analytics` creados en BigQuery).

---
## 1. Setup del entorno híbrido

Mismo patrón que el Módulo 1: detección automática de Vertex AI Workbench vs local.

In [1]:
#%pip install google-cloud-bigquery-analyticshub

In [2]:
# Instalar dependencias (descomentar en primera ejecución)
# !pip install google-cloud-bigquery google-cloud-storage google-cloud-bigquery-connection pandas pyarrow db-dtypes python-dotenv

import os
import pandas as pd
import json
from google.cloud import bigquery, storage
from google.api_core.exceptions import NotFound, Conflict
import warnings
warnings.filterwarnings('ignore')

# --- Detección de entorno ---
IN_VERTEX_AI = any([
    os.environ.get("DL_ANACONDA_HOME"),
    os.path.exists("/opt/deeplearning/metadata"),
    os.environ.get("GOOGLE_CLOUD_PROJECT"),
])

if IN_VERTEX_AI:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "people-analytics-formacion")
    bq_client = bigquery.Client(project=PROJECT_ID)
    gcs_client = storage.Client(project=PROJECT_ID)
    print(f"Entorno: Vertex AI Workbench (ADC)")
else:
    from dotenv import load_dotenv
    from google.oauth2 import service_account
    load_dotenv()
    PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "people-analytics-formacion")
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "service-account.json")
    if os.path.exists(creds_path):
        credentials = service_account.Credentials.from_service_account_file(creds_path)
        bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        gcs_client = storage.Client(project=PROJECT_ID, credentials=credentials)
    else:
        bq_client = bigquery.Client(project=PROJECT_ID)
        gcs_client = storage.Client(project=PROJECT_ID)
    print(f"Entorno: Local")

REGION = "europe-southwest1"
BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", f"{PROJECT_ID}-datalake")

print(f"Proyecto: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Cliente BQ: {bq_client.project}")
print(f"Cliente GCS: {gcs_client.project}")

Entorno: Local
Proyecto: project-9176af0b-ecb3-4050-859
Region: europe-southwest1
Bucket: project-9176af0b-ecb3-4050-859-datalake
Cliente BQ: project-9176af0b-ecb3-4050-859
Cliente GCS: project-9176af0b-ecb3-4050-859


---
## 2. Exploración de la arquitectura Medallion (INFORMATION_SCHEMA)

En el Módulo 1 creamos 4 datasets con arquitectura Medallion. Ahora usamos las vistas del sistema
de BigQuery (`INFORMATION_SCHEMA`) para explorar qué construimos: tablas, esquemas, particionado, clustering.

Esto es lo que un ingeniero de datos hace al heredar un proyecto o auditar una arquitectura existente.

In [3]:
# --- Inventario de datasets y tablas ---
print("DATASETS EN EL PROYECTO")
print("=" * 70)

for ds in bq_client.list_datasets():
    dataset_ref = bq_client.get_dataset(ds.reference)
    tables = list(bq_client.list_tables(ds.reference))
    print(f"\n  {ds.dataset_id} ({dataset_ref.location})")
    print(f"    Descripción: {dataset_ref.description or '(sin descripción)'}")
    print(f"    Tablas: {len(tables)}")
    for t in tables:
        table = bq_client.get_table(t.reference)
        partition_info = f" | partition: {table.time_partitioning.field}" if table.time_partitioning else ""
        cluster_info = f" | cluster: {', '.join(table.clustering_fields)}" if table.clustering_fields else ""
        print(f"      - {t.table_id}: {table.num_rows} filas, {len(table.schema)} cols{partition_info}{cluster_info}")

DATASETS EN EL PROYECTO

  bronze_personio (europe-southwest1)
    Descripción: Capa Bronze — datos crudos de Personio sin transformar
    Tablas: 4
      - ext_employee_parquet: 0 filas, 51 cols
      - raw_employee_data: 209 filas, 117 cols
      - raw_employee_history: 1189 filas, 79 cols
      - raw_gross_salary: 994 filas, 109 cols

  bronze_personio_dev (europe-southwest1)
    Descripción: DEV — capa Bronze para desarrollo y experimentación
    Tablas: 0

  gold_people_analytics (europe-southwest1)
    Descripción: Capa Gold — métricas de negocio para People Analytics
    Tablas: 4
      - compensation_analysis: 121 filas, 19 cols
      - headcount_monthly: 317 filas, 14 cols
      - turnover_metrics: 209 filas, 18 cols
      - v_headcount_resumen: 0 filas, 6 cols

  gold_people_analytics_dev (europe-southwest1)
    Descripción: DEV — capa Gold para validación de métricas
    Tablas: 0

  ml_features (europe-southwest1)
    Descripción: Feature store para modelos de ML (Vertex AI

In [4]:
# --- INFORMATION_SCHEMA: explorar esquemas de tablas Silver ---
sql_schemas = f"""
SELECT
    table_name,
    column_name,
    data_type,
    is_nullable
FROM `{PROJECT_ID}.silver_personio.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'dim_employee'
ORDER BY ordinal_position
"""
df_schema = bq_client.query(sql_schemas).to_dataframe()

print("ESQUEMA DE dim_employee (Silver)")
print("=" * 70)
print(f"Total columnas: {len(df_schema)}")
print(f"\nTipos de datos:")
print(df_schema['data_type'].value_counts().to_string())
print(f"\nColumnas:")
for _, row in df_schema.iterrows():
    nullable = "NULL" if row['is_nullable'] == 'YES' else "NOT NULL"
    print(f"  {row['column_name']:<35} {row['data_type']:<15} {nullable}")

ESQUEMA DE dim_employee (Silver)
Total columnas: 51

Tipos de datos:
data_type
STRING     36
DATE        5
FLOAT64     5
INT64       4
BOOL        1

Columnas:
  employee_code                       INT64           NULL
  email                               STRING          NULL
  first_name                          STRING          NULL
  last_name                           STRING          NULL
  gender                              STRING          NULL
  hire_date                           DATE            NULL
  termination_date                    DATE            NULL
  termination_type                    STRING          NULL
  termination_reason                  STRING          NULL
  status                              STRING          NULL
  department                          STRING          NULL
  team                                STRING          NULL
  subteam                             STRING          NULL
  position                            STRING          NULL
  role        

In [5]:
# --- INFORMATION_SCHEMA: particionado y clustering ---
sql_partitions = f"""
SELECT
    table_catalog,
    table_schema,
    table_name,
    partition_id,
    total_rows,
    total_logical_bytes,
    ROUND(total_logical_bytes / 1024 / 1024, 2) AS size_mb,
    last_modified_time
FROM `{PROJECT_ID}.silver_personio.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = 'fact_salary_history'
ORDER BY partition_id
"""
df_partitions = bq_client.query(sql_partitions).to_dataframe()

print("PARTICIONES DE fact_salary_history")
print("=" * 70)
print(f"Total particiones: {len(df_partitions)}")
print(f"Total filas: {df_partitions['total_rows'].sum():,}")
print(f"Tamaño total: {df_partitions['size_mb'].sum():.2f} MB")
print(f"\nDetalle por partición:")
for _, row in df_partitions.iterrows():
    print(f"  {row['partition_id']}: {row['total_rows']:>6} filas, {row['size_mb']:>6.2f} MB")

PARTICIONES DE fact_salary_history
Total particiones: 12
Total filas: 1,189
Tamaño total: 0.36 MB

Detalle por partición:
  20250101:     91 filas,   0.03 MB
  20250201:     91 filas,   0.03 MB
  20250301:     90 filas,   0.03 MB
  20250401:     91 filas,   0.03 MB
  20250501:     93 filas,   0.03 MB
  20250601:     95 filas,   0.03 MB
  20250701:    100 filas,   0.03 MB
  20250801:    100 filas,   0.03 MB
  20250901:    107 filas,   0.03 MB
  20251001:    109 filas,   0.03 MB
  20251101:    111 filas,   0.03 MB
  20251201:    111 filas,   0.03 MB


In [6]:
# --- Almacenamiento por capa: comparación Bronze vs Silver vs Gold ---
# `region-*.INFORMATION_SCHEMA.TABLE_STORAGE` requiere `bigquery.tables.list`
# a nivel proyecto (rol `roles/bigquery.resourceViewer` o superior), que
# no siempre está disponible para cuentas de servicio de curso.
# Alternativa robusta: usar el SDK (`get_table`), que solo requiere lectura
# a nivel dataset y expone `num_bytes` (lógico) y `num_rows`.

datasets_to_check = ["bronze_personio", "silver_personio", "gold_people_analytics"]
rows = []
for dataset_id in datasets_to_check:
    for t in bq_client.list_tables(f"{PROJECT_ID}.{dataset_id}"):
        table = bq_client.get_table(t.reference)
        # Solo tablas nativas tienen almacenamiento propio (las EXTERNAL viven en GCS)
        if table.table_type != "TABLE":
            continue
        logical_mb = (table.num_bytes or 0) / 1024 / 1024
        rows.append({
            "dataset": dataset_id,
            "table_name": t.table_id,
            "logical_mb": round(logical_mb, 2),
            "total_rows": table.num_rows or 0,
        })

df_storage = pd.DataFrame(rows)

print("ALMACENAMIENTO POR CAPA (bytes lógicos)")
print("=" * 75)
print(f"{'Dataset':<28} {'Tabla':<30} {'Lógico MB':>10} {'Filas':>10}")
print("-" * 75)
for _, row in df_storage.iterrows():
    print(f"  {row['dataset']:<26} {row['table_name']:<30} {row['logical_mb']:>10.2f} {row['total_rows']:>10}")

# Resumen por capa
print(f"\nResumen por capa:")
summary = df_storage.groupby("dataset").agg({"logical_mb": "sum", "total_rows": "sum"}).round(2)
for ds, row in summary.iterrows():
    print(f"  {ds:<30} {row['logical_mb']:>8.2f} MB lógico  |  {int(row['total_rows']):>6} filas")

print(f"\nNota: para comparar bytes lógicos vs físicos (compresión Capacitor)")
print(f"se requiere `INFORMATION_SCHEMA.TABLE_STORAGE` a nivel proyecto,")
print(f"que exige el rol `roles/bigquery.resourceViewer` (o superior).")

ALMACENAMIENTO POR CAPA (bytes lógicos)
Dataset                      Tabla                           Lógico MB      Filas
---------------------------------------------------------------------------
  bronze_personio            raw_employee_data                    0.17        209
  bronze_personio            raw_employee_history                 0.60       1189
  bronze_personio            raw_gross_salary                     0.85        994
  silver_personio            dim_employee                         0.11        209
  silver_personio            fact_payroll_monthly                 0.36        994
  silver_personio            fact_salary_history                  0.36       1189
  gold_people_analytics      compensation_analysis                0.02        121
  gold_people_analytics      headcount_monthly                    0.04        317
  gold_people_analytics      turnover_metrics                     0.04        209

Resumen por capa:
  bronze_personio                    1.62 MB 

---
## 3. Cloud Storage como Data Lake

En una arquitectura profesional, **Cloud Storage (GCS)** complementa a BigQuery:
- **GCS** = Data Lake (archivos crudos, backups, datos no estructurados, archivos grandes)
- **BigQuery** = Data Warehouse (consultas SQL, métricas, ML)

Creamos un bucket con estructura medallion en GCS y subimos los datos crudos.
Esto simula el patrón real: los datos llegan primero a GCS (landing zone) y luego se cargan a BigQuery.

```
gs://{proyecto}-datalake/
├── bronze/
│   ├── personio/raw/YYYY-MM-DD/
│   ├── nomina/raw/YYYY-MM/
│   └── encuestas/raw/YYYY-QN/
├── silver/
│   └── personio/parquet/
├── gold/
│   └── exports/
└── archive/
    └── YYYY/
```

In [7]:
# --- Crear bucket (idempotente) ---
from google.cloud.storage import Bucket

try:
    bucket = gcs_client.get_bucket(BUCKET_NAME)
    print(f"Bucket existe: gs://{BUCKET_NAME}")
except NotFound:
    bucket = Bucket(gcs_client, BUCKET_NAME)
    bucket.location = REGION
    bucket.storage_class = "STANDARD"
    # Lifecycle: mover a Nearline después de 90 días, Archive después de 365
    bucket.add_lifecycle_delete_rule(age=730)
    bucket.add_lifecycle_set_storage_class_rule("NEARLINE", age=90)
    bucket.add_lifecycle_set_storage_class_rule("ARCHIVE", age=365)
    bucket = gcs_client.create_bucket(bucket)
    print(f"Bucket creado: gs://{BUCKET_NAME}")

print(f"  Location: {bucket.location}")
print(f"  Storage class: {bucket.storage_class}")
print(f"  Lifecycle rules: {len(list(bucket.lifecycle_rules))}")

Bucket existe: gs://project-9176af0b-ecb3-4050-859-datalake
  Location: EUROPE-SOUTHWEST1
  Storage class: STANDARD
  Lifecycle rules: 3


In [8]:
# --- Subir CSVs crudos a la capa Bronze de GCS ---
from datetime import date

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "documentacion")
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), "data", "documentacion")
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), "..", "data", "documentacion")

snapshot_date = date.today().isoformat()

csv_files = {
    "personio_data_v2_anon.csv": f"bronze/personio/raw/{snapshot_date}/employee_data.csv",
    "personio_history_v2_anon.csv": f"bronze/personio/raw/{snapshot_date}/employee_history.csv",
    "gross_salary_v2_anon.csv": f"bronze/nomina/raw/{snapshot_date}/gross_salary.csv",
}

print(f"Subiendo datos a gs://{BUCKET_NAME}/ (snapshot: {snapshot_date})")
for local_name, gcs_path in csv_files.items():
    local_path = os.path.join(DATA_DIR, local_name)
    if os.path.exists(local_path):
        blob = bucket.blob(gcs_path)
        blob.upload_from_filename(local_path)
        size_kb = os.path.getsize(local_path) / 1024
        print(f"  {gcs_path} ({size_kb:.0f} KB)")
    else:
        print(f"  SKIP: {local_name} no encontrado en {DATA_DIR}")

# --- Exportar tabla Silver a Parquet en GCS ---
print(f"\nExportando Silver a Parquet en GCS...")
for table_name in ["dim_employee", "fact_salary_history", "fact_payroll_monthly"]:
    destination_uri = f"gs://{BUCKET_NAME}/silver/personio/parquet/{table_name}/*.parquet"
    table_ref = f"{PROJECT_ID}.silver_personio.{table_name}"

    extract_job = bq_client.extract_table(
        table_ref,
        destination_uri,
        job_config=bigquery.ExtractJobConfig(destination_format="PARQUET"),
    )
    extract_job.result()
    print(f"  silver/personio/parquet/{table_name}/")

print(f"\nData Lake configurado en gs://{BUCKET_NAME}/")

Subiendo datos a gs://project-9176af0b-ecb3-4050-859-datalake/ (snapshot: 2026-04-20)
  bronze/personio/raw/2026-04-20/employee_data.csv (167 KB)
  bronze/personio/raw/2026-04-20/employee_history.csv (587 KB)
  bronze/nomina/raw/2026-04-20/gross_salary.csv (594 KB)

Exportando Silver a Parquet en GCS...
  silver/personio/parquet/dim_employee/
  silver/personio/parquet/fact_salary_history/
  silver/personio/parquet/fact_payroll_monthly/

Data Lake configurado en gs://project-9176af0b-ecb3-4050-859-datalake/


In [9]:
# --- Inventario del Data Lake ---
print(f"CONTENIDO DEL DATA LAKE: gs://{BUCKET_NAME}/")
print("=" * 70)

total_size = 0
blobs_by_prefix = {}

for blob in bucket.list_blobs():
    prefix = "/".join(blob.name.split("/")[:2])
    if prefix not in blobs_by_prefix:
        blobs_by_prefix[prefix] = {"count": 0, "size": 0}
    blobs_by_prefix[prefix]["count"] += 1
    blobs_by_prefix[prefix]["size"] += blob.size
    total_size += blob.size

for prefix, info in sorted(blobs_by_prefix.items()):
    size_kb = info["size"] / 1024
    print(f"  {prefix}/ — {info['count']} archivos, {size_kb:.0f} KB")

print(f"\nTotal: {sum(i['count'] for i in blobs_by_prefix.values())} archivos, {total_size / 1024:.0f} KB")
print(f"\nEstructura Data Lake vs Data Warehouse:")
print(f"  GCS (Data Lake):  archivos crudos (CSV, Parquet) para landing zone y backup")
print(f"  BQ (Warehouse):   tablas estructuradas con SQL, particionado, clustering")

CONTENIDO DEL DATA LAKE: gs://project-9176af0b-ecb3-4050-859-datalake/
  bronze/nomina/ — 2 archivos, 1188 KB
  bronze/personio/ — 4 archivos, 1508 KB
  silver/personio/ — 27 archivos, 665 KB

Total: 33 archivos, 3361 KB

Estructura Data Lake vs Data Warehouse:
  GCS (Data Lake):  archivos crudos (CSV, Parquet) para landing zone y backup
  BQ (Warehouse):   tablas estructuradas con SQL, particionado, clustering


---
## 4. BigLake — Tablas externas sobre el Data Lake

**BigLake** unifica el acceso a datos en GCS y BigQuery bajo una misma capa de seguridad.
Permite consultar archivos Parquet/CSV en Cloud Storage directamente con SQL de BigQuery,
sin necesidad de cargarlos primero. Los permisos IAM se aplican de forma unificada.

Ventajas:
- **Sin duplicación de datos** — los Parquet quedan en GCS, BigQuery los lee directamente
- **Permisos unificados** — row-level y column-level security funcionan igual que en tablas nativas
- **Ideal para datos históricos** — archivos que no necesitan estar cargados permanentemente en BQ

In [10]:
# --- Crear tabla externa sobre Parquet en GCS ---
# Opción 1: External table estándar (sin BigLake connection)
# Funciona sin configurar una connection, ideal para empezar

EXTERNAL_DATASET = "bronze_personio"

sql_external_table = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{EXTERNAL_DATASET}.ext_employee_parquet`
OPTIONS (
    format = 'PARQUET',
    uris = ['gs://{BUCKET_NAME}/silver/personio/parquet/dim_employee/*.parquet']
)
"""

bq_client.query(sql_external_table).result()
print("Tabla externa creada: ext_employee_parquet")

# Consultar la tabla externa — BigQuery lee directamente del Parquet en GCS
sql_test_external = f"""
SELECT
    employee_code,
    gender,
    department,
    country,
    band,
    gross_salary_annual,
    tenure_months
FROM `{PROJECT_ID}.{EXTERNAL_DATASET}.ext_employee_parquet`
WHERE is_active = TRUE
ORDER BY gross_salary_annual DESC
LIMIT 10
"""

df_external = bq_client.query(sql_test_external).to_dataframe()
print(f"\nTop 10 salarios (desde Parquet en GCS, sin cargar a BQ):")
print(df_external.to_string(index=False))

Tabla externa creada: ext_employee_parquet

Top 10 salarios (desde Parquet en GCS, sin cargar a BQ):
 employee_code gender         department       country band  gross_salary_annual  tenure_months
          6618 female             People     Argentina None          24343464.88              3
          6118 female PR & Communication         Japan None           4711759.12              5
          6742 female PR & Communication         India None           2742051.48              2
          4690 female             People        Mexico   P2            366595.14             17
          5947 female PR & Communication        Brazil   P2            113756.06              6
          6274 female PR & Communication United States None            112559.92              3
           741   male PR & Communication United States   M2            112054.37             28
          2057 female PR & Communication     Australia   P3            104320.60             44
          2008   male             P

In [11]:
# --- Comparar: tabla nativa vs tabla externa ---
# La tabla nativa (en BQ) y la externa (sobre GCS) deben dar el mismo resultado

sql_compare = f"""
WITH native AS (
    SELECT COUNT(*) AS n, AVG(gross_salary_annual) AS avg_salary
    FROM `{PROJECT_ID}.silver_personio.dim_employee`
    WHERE is_active = TRUE
),
external AS (
    SELECT COUNT(*) AS n, AVG(gross_salary_annual) AS avg_salary
    FROM `{PROJECT_ID}.{EXTERNAL_DATASET}.ext_employee_parquet`
    WHERE is_active = TRUE
)
SELECT
    'Tabla nativa (BQ)' AS fuente, n, ROUND(avg_salary, 2) AS avg_salary FROM native
UNION ALL
SELECT
    'Tabla externa (GCS)' AS fuente, n, ROUND(avg_salary, 2) AS avg_salary FROM external
"""

df_compare = bq_client.query(sql_compare).to_dataframe()
print("COMPARACIÓN: Tabla nativa vs Tabla externa")
print("=" * 60)
print(df_compare.to_string(index=False))
print(f"\nAmbas fuentes devuelven los mismos datos.")
print(f"La diferencia: la tabla nativa es más rápida (datos en Capacitor),")
print(f"la tabla externa ahorra almacenamiento en BQ (datos en GCS).")

COMPARACIÓN: Tabla nativa vs Tabla externa
             fuente   n  avg_salary
  Tabla nativa (BQ) 121   307463.77
Tabla externa (GCS) 121   307463.77

Ambas fuentes devuelven los mismos datos.
La diferencia: la tabla nativa es más rápida (datos en Capacitor),
la tabla externa ahorra almacenamiento en BQ (datos en GCS).


---
## 5. Auditoría de jobs y control de costes (INFORMATION_SCHEMA.JOBS)

`INFORMATION_SCHEMA.JOBS` registra cada query ejecutada en el proyecto.
Es la base para **FinOps** (control de costes) y **auditoría** (quién consultó qué).

En People Analytics esto es crítico:
- ¿Quién está consultando datos de salarios?
- ¿Cuánto cuestan nuestras queries mensuales?
- ¿Hay queries ineficientes que escanean tablas enteras?

In [12]:
# --- Historial de jobs recientes ---
# `region-*.INFORMATION_SCHEMA.JOBS` requiere `bigquery.jobs.listAll` a nivel
# proyecto (rol `roles/bigquery.resourceViewer` o `roles/bigquery.admin`), que
# normalmente solo tiene el administrador del proyecto.
#
# Para usuarios normales usamos `JOBS_BY_USER`, que solo expone los jobs
# lanzados por el propio caller y únicamente requiere `bigquery.jobs.list`
# (permiso que ya tiene cualquier usuario que ejecute queries).

sql_jobs = f"""
SELECT
    job_id,
    job_type,
    user_email,
    state,
    creation_time,
    end_time,
    TIMESTAMP_DIFF(end_time, start_time, SECOND) AS duration_seconds,
    total_bytes_processed,
    ROUND(total_bytes_processed / 1024 / 1024, 2) AS mb_processed,
    total_bytes_billed,
    ROUND(total_bytes_billed / POW(1024, 4) * 6.25, 4) AS estimated_cost_usd,
    cache_hit,
    statement_type
FROM `{PROJECT_ID}.region-{REGION}.INFORMATION_SCHEMA.JOBS_BY_USER`
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
ORDER BY creation_time DESC
LIMIT 20
"""

df_jobs = bq_client.query(sql_jobs).to_dataframe()


def _fmt(val, spec):
    """Formatea valores que pueden venir como pd.NA (comunes en jobs LOAD/EXTRACT)."""
    return format(val, spec) if pd.notna(val) else "-"


print("ÚLTIMOS 20 JOBS DEL USUARIO ACTUAL (últimas 24h)")
print("=" * 90)
print(f"{'Tipo':<12} {'Statement':<15} {'MB proc':>10} {'Coste $':>10} {'Cache':>6} {'Duración':>10}")
print("-" * 90)
for _, row in df_jobs.iterrows():
    cache_val = row.get("cache_hit")
    if pd.notna(cache_val):
        cache = "Sí" if bool(cache_val) else "No"
    else:
        cache = "-"
    dur_val = row.get("duration_seconds")
    duration = f"{dur_val:.0f}s" if pd.notna(dur_val) else "-"
    stmt = str(row.get("statement_type") or "-")[:14]
    mb = _fmt(row.get("mb_processed"), ".2f")
    cost = _fmt(row.get("estimated_cost_usd"), ".4f")
    print(f"  {row['job_type']:<10} {stmt:<15} {mb:>10} {cost:>10} {cache:>6} {duration:>10}")

print(f"\nNota: `JOBS_BY_USER` solo muestra los jobs del usuario actual.")
print(f"Para auditoría a nivel proyecto (todos los usuarios) se usa `JOBS`,")
print(f"que requiere `roles/bigquery.resourceViewer` o `roles/bigquery.admin`.")

ÚLTIMOS 20 JOBS DEL USUARIO ACTUAL (últimas 24h)
Tipo         Statement          MB proc    Coste $  Cache   Duración
------------------------------------------------------------------------------------------
  QUERY      SELECT                0.00     0.0001     No         0s
  QUERY      SELECT                0.01     0.0001     No         0s
  QUERY      CREATE_EXTERNA        0.00     0.0000     No         0s
  EXTRACT    -                     0.36          -      -         0s
  EXTRACT    -                     0.36          -      -         0s
  EXTRACT    -                     0.11          -      -         0s
  QUERY      SELECT               10.00     0.0001     No         0s
  QUERY      SELECT               10.00     0.0001     No         0s
  QUERY      CREATE_TABLE_A        0.11     0.0001     No         1s
  QUERY      CREATE_VIEW           0.00     0.0000     No         0s
  QUERY      SELECT                0.17     0.0001     No         0s
  QUERY      SELECT             

In [13]:
# --- Coste del usuario actual (últimos 7 días) ---
# Con `JOBS_BY_USER` solo vemos los jobs del caller, así que el "coste por
# usuario" se convierte en "coste propio". En producción, con el rol
# `roles/bigquery.resourceViewer`, se sustituye por `JOBS` y se agrupa por
# `user_email` para tener la vista completa del proyecto.

sql_cost_by_user = f"""
SELECT
    user_email,
    COUNT(*) AS num_queries,
    ROUND(SUM(total_bytes_processed) / 1024 / 1024, 2) AS total_mb_processed,
    ROUND(SUM(total_bytes_billed) / POW(1024, 4) * 6.25, 4) AS total_cost_usd,
    COUNTIF(cache_hit) AS cache_hits,
    ROUND(SAFE_DIVIDE(COUNTIF(cache_hit), COUNT(*)) * 100, 1) AS cache_hit_pct
FROM `{PROJECT_ID}.region-{REGION}.INFORMATION_SCHEMA.JOBS_BY_USER`
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
    AND job_type = 'QUERY'
    AND state = 'DONE'
GROUP BY user_email
ORDER BY total_cost_usd DESC
"""

df_cost_user = bq_client.query(sql_cost_by_user).to_dataframe()

print("COSTE DEL USUARIO ACTUAL (últimos 7 días)")
print("=" * 80)
for _, row in df_cost_user.iterrows():
    print(f"  {row['user_email']}")
    print(f"    Queries: {row['num_queries']}  |  MB procesados: {row['total_mb_processed']:.2f}  |  Coste: ${row['total_cost_usd']:.4f}")
    print(f"    Cache hits: {row['cache_hits']} ({row['cache_hit_pct']:.1f}%)")

# --- Queries propias que accedieron a datos de compensación ---
sql_salary_access = f"""
SELECT
    user_email,
    creation_time,
    SUBSTR(query, 1, 120) AS query_preview
FROM `{PROJECT_ID}.region-{REGION}.INFORMATION_SCHEMA.JOBS_BY_USER`
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
    AND job_type = 'QUERY'
    AND state = 'DONE'
    AND (
        LOWER(query) LIKE '%salary%'
        OR LOWER(query) LIKE '%salario%'
        OR LOWER(query) LIKE '%compensation%'
        OR LOWER(query) LIKE '%payroll%'
    )
ORDER BY creation_time DESC
LIMIT 10
"""

df_salary = bq_client.query(sql_salary_access).to_dataframe()

print(f"\nAUDITORÍA: Queries propias que accedieron a datos de compensación")
print("=" * 80)
if len(df_salary) > 0:
    for _, row in df_salary.iterrows():
        print(f"  [{row['creation_time']}] {row['user_email']}")
        print(f"    {row['query_preview']}...")
else:
    print("  No se encontraron queries de compensación en los últimos 7 días")

print(f"\nEn producción, el equipo de gobierno usa `JOBS` (no `JOBS_BY_USER`)")
print(f"para auditar accesos de todos los usuarios a tablas sensibles.")

COSTE DEL USUARIO ACTUAL (últimos 7 días)
  carlos.g.chou@gmail.com
    Queries: 99  |  MB procesados: 144.03  |  Coste: $0.0054
    Cache hits: 0 (0.0%)

AUDITORÍA: Queries propias que accedieron a datos de compensación
  [2026-04-20 09:19:47.525000+00:00] carlos.g.chou@gmail.com
    
WITH native AS (
    SELECT COUNT(*) AS n, AVG(gross_salary_annual) AS avg_salary
    FROM `project-9176af0b-ecb3-4050-...
  [2026-04-20 09:19:46.875000+00:00] carlos.g.chou@gmail.com
    
SELECT
    employee_code,
    gender,
    department,
    country,
    band,
    gross_salary_annual,
    tenure_months...
  [2026-04-20 09:19:37.565000+00:00] carlos.g.chou@gmail.com
    
SELECT
    table_catalog,
    table_schema,
    table_name,
    partition_id,
    total_rows,
    total_logical_bytes,
...
  [2026-04-20 08:34:23.253000+00:00] carlos.g.chou@gmail.com
    
CREATE OR REPLACE VIEW `project-9176af0b-ecb3-4050-859.gold_people_analytics.v_headcount_resumen`
AS
SELECT
    snapsho...
  [2026-04-20 08:34:21.

---
## 6. IAM y seguridad a nivel dataset

En People Analytics, la seguridad no es opcional. Los datos de salarios, género y evaluaciones
son categorías especiales bajo GDPR (Art. 9). IAM controla **quién** puede hacer **qué** sobre **qué recurso**.

Principios:
- **Mínimo privilegio** — cada persona y service account solo tiene los permisos estrictamente necesarios
- **Grupos, no personas** — asignar permisos a grupos (`pa-analysts@`, `pa-engineers@`)
- **Separación por capa** — RRHH solo ve Gold, ingenieros ven Bronze/Silver
- **Service accounts por función** — `sa-ingesta@`, `sa-dataform@`, `sa-looker@`

In [14]:
# --- Consultar permisos actuales de cada dataset ---
print("PERMISOS IAM POR DATASET")
print("=" * 80)

for dataset_id in ["bronze_personio", "silver_personio", "gold_people_analytics", "ml_features"]:
    dataset_ref = bq_client.get_dataset(f"{PROJECT_ID}.{dataset_id}")
    entries = dataset_ref.access_entries

    print(f"\n  {dataset_id}:")
    for entry in entries:
        role = entry.role or "(inherited)"
        entity_type = entry.entity_type or "special"
        entity_id = entry.entity_id or "(project default)"
        print(f"    {role:<25} {entity_type:<15} {entity_id}")

PERMISOS IAM POR DATASET

  bronze_personio:
    WRITER                    specialGroup    projectWriters
    OWNER                     specialGroup    projectOwners
    OWNER                     userByEmail     carlos.g.chou@gmail.com
    READER                    specialGroup    projectReaders

  silver_personio:
    WRITER                    specialGroup    projectWriters
    OWNER                     specialGroup    projectOwners
    OWNER                     userByEmail     carlos.g.chou@gmail.com
    READER                    specialGroup    projectReaders

  gold_people_analytics:
    WRITER                    specialGroup    projectWriters
    OWNER                     specialGroup    projectOwners
    OWNER                     userByEmail     carlos.g.chou@gmail.com
    READER                    specialGroup    projectReaders

  ml_features:
    WRITER                    specialGroup    projectWriters
    OWNER                     specialGroup    projectOwners
    OWNER       

In [15]:
# --- Crear una vista autorizada (authorized view) ---
# Las vistas autorizadas permiten que RRHH vea métricas Gold
# sin tener acceso directo a los datos Silver/Bronze subyacentes

sql_authorized_view = f"""
CREATE OR REPLACE VIEW `{PROJECT_ID}.gold_people_analytics.v_headcount_resumen`
AS
SELECT
    snapshot_date,
    department,
    hc_status,
    SUM(headcount) AS total_headcount,
    ROUND(SUM(total_fte), 1) AS total_fte,
    ROUND(AVG(avg_fixed_usd), 0) AS avg_fixed_salary_usd
FROM `{PROJECT_ID}.gold_people_analytics.headcount_monthly`
GROUP BY snapshot_date, department, hc_status
"""

bq_client.query(sql_authorized_view).result()
print("Vista creada: gold_people_analytics.v_headcount_resumen")
print(f"\nEsta vista se puede compartir con RRHH sin dar acceso al dataset Silver.")
print(f"RRHH ejecuta: SELECT * FROM v_headcount_resumen WHERE department = 'People'")
print(f"Y solo ve datos agregados, nunca datos individuales de empleados.")

# --- Simulación de política de acceso por capa ---
print(f"\n{'=' * 70}")
print("POLÍTICA DE ACCESO RECOMENDADA POR CAPA")
print("=" * 70)

policy = {
    "bronze_personio": {
        "sa-ingesta@": "WRITER (carga datos crudos)",
        "pa-engineers@": "READER (debugging)",
        "pa-analysts@": "SIN ACCESO",
        "pa-rrhh@": "SIN ACCESO",
    },
    "silver_personio": {
        "sa-dataform@": "WRITER (transformaciones)",
        "pa-engineers@": "READER",
        "pa-analysts@": "SIN ACCESO",
        "pa-rrhh@": "SIN ACCESO",
    },
    "gold_people_analytics": {
        "sa-looker@": "READER (dashboards)",
        "pa-engineers@": "READER",
        "pa-analysts@": "READER (solo vistas)",
        "pa-rrhh@": "READER (solo vistas autorizadas)",
    },
    "ml_features": {
        "sa-vertexai@": "WRITER (feature store)",
        "pa-engineers@": "READER",
        "pa-analysts@": "READER",
        "pa-rrhh@": "SIN ACCESO",
    },
}

for dataset, roles in policy.items():
    print(f"\n  {dataset}:")
    for principal, access in roles.items():
        print(f"    {principal:<25} → {access}")

Vista creada: gold_people_analytics.v_headcount_resumen

Esta vista se puede compartir con RRHH sin dar acceso al dataset Silver.
RRHH ejecuta: SELECT * FROM v_headcount_resumen WHERE department = 'People'
Y solo ve datos agregados, nunca datos individuales de empleados.

POLÍTICA DE ACCESO RECOMENDADA POR CAPA

  bronze_personio:
    sa-ingesta@               → WRITER (carga datos crudos)
    pa-engineers@             → READER (debugging)
    pa-analysts@              → SIN ACCESO
    pa-rrhh@                  → SIN ACCESO

  silver_personio:
    sa-dataform@              → WRITER (transformaciones)
    pa-engineers@             → READER
    pa-analysts@              → SIN ACCESO
    pa-rrhh@                  → SIN ACCESO

  gold_people_analytics:
    sa-looker@                → READER (dashboards)
    pa-engineers@             → READER
    pa-analysts@              → READER (solo vistas)
    pa-rrhh@                  → READER (solo vistas autorizadas)

  ml_features:
    sa-vertexai@

---
## 7. Analytics Hub — Compartición segura de datos

**Analytics Hub** permite publicar datasets curados para que otros departamentos los consuman
sin copiar datos ni dar acceso directo a BigQuery.

```
Equipo de Datos (publisher)         RRHH (subscriber)         Finanzas (subscriber)
┌──────────────────────────┐       ┌─────────────────┐       ┌─────────────────┐
│ gold_people_analytics    │       │ linked_dataset   │       │ linked_dataset   │
│   headcount_monthly      │publish│   (solo lectura) │       │   (solo lectura) │
│   compensation_analysis  │──────►│   ve headcount   │       │   ve coste       │
│   turnover_metrics       │       │   y rotación     │       │   laboral        │
└──────────────────────────┘       └─────────────────┘       └─────────────────┘
```

Ventajas sobre IAM directo:
- **Catálogo centralizado** — los consumidores descubren datasets disponibles
- **Gobernanza** — aprobaciones, auditoría, revocación centralizada
- **Cross-org** — compartir con otras filiales sin configuración compleja
- **El suscriptor paga** — cada departamento paga sus propias queries

In [16]:
# --- Analytics Hub: crear Exchange y Listing ---
# Requiere:
#   1. Librería cliente: pip install google-cloud-bigquery-analyticshub
#   2. API habilitada: gcloud services enable analyticshub.googleapis.com
#   3. Rol IAM: roles/analyticshub.admin (o permisos equivalentes)

EXCHANGE_ID = "people_analytics_exchange"
LISTING_ID = "gold_metrics"

try:
    from google.cloud import bigquery_analyticshub_v1 as analyticshub
    from google.api_core.exceptions import AlreadyExists, PermissionDenied
except ImportError:
    print("Falta la librería `google-cloud-bigquery-analyticshub`.")
    print("Instálala con:")
    print("    pip install google-cloud-bigquery-analyticshub")

    print("  BigQuery → Analytics Hub → Create Exchange → Add Listing → gold_people_analytics")
else:
    try:
        ah_client = analyticshub.AnalyticsHubServiceClient()

        # 1. Crear Data Exchange (catálogo)
        parent = f"projects/{PROJECT_ID}/locations/{REGION}"
        exchange = analyticshub.DataExchange(
            display_name="People Analytics — Métricas Gold",
            description="Métricas curadas de People Analytics para consumo por RRHH, Finanzas y Dirección",
            primary_contact="people-analytics@empresa.com",
        )

        try:
            created = ah_client.create_data_exchange(
                parent=parent,
                data_exchange_id=EXCHANGE_ID,
                data_exchange=exchange,
            )
            print(f"Exchange creado: {created.name}")
        except AlreadyExists:
            print(f"Exchange ya existe: {EXCHANGE_ID}")
            created = ah_client.get_data_exchange(
                name=f"{parent}/dataExchanges/{EXCHANGE_ID}"
            )

        # 2. Crear Listing (dataset compartido)
        listing = analyticshub.Listing(
            display_name="Métricas Gold de People Analytics",
            description="Headcount, compensación y rotación — actualizado diariamente",
            bigquery_dataset=analyticshub.Listing.BigQueryDatasetSource(
                dataset=f"projects/{PROJECT_ID}/datasets/gold_people_analytics"
            ),
            categories=["CATEGORY_OTHERS"],
        )

        try:
            created_listing = ah_client.create_listing(
                parent=created.name,
                listing_id=LISTING_ID,
                listing=listing,
            )
            print(f"Listing creado: {created_listing.name}")
        except AlreadyExists:
            print(f"Listing ya existe: {LISTING_ID}")

        print(f"Los suscriptores pueden buscar '{EXCHANGE_ID}' en Analytics Hub")
        print(f"y suscribirse para recibir un linked dataset en su proyecto.")

    except PermissionDenied:
        print("Analytics Hub: permisos insuficientes (necesita analyticshub.dataExchanges.create)")
        print("Solicitar el rol: roles/analyticshub.admin al administrador del proyecto")
        print("  BigQuery → Analytics Hub → Create Exchange → Add Listing → gold_people_analytics")
    except Exception as e:
        print(f"Analytics Hub no disponible: {type(e).__name__}: {e}")
        print("Asegurar que la API está habilitada: gcloud services enable analyticshub.googleapis.com")

Analytics Hub no disponible: TypeError: AnalyticsHubServiceClient.create_data_exchange() got an unexpected keyword argument 'data_exchange_id'. Did you mean 'data_exchange'?
Asegurar que la API está habilitada: gcloud services enable analyticshub.googleapis.com


---
## 8. Separación dev/test/prod — Patrones en BigQuery

En producción real, los datos de empleados requieren aislamiento estricto.
GDPR Art. 25 (*Data Protection by Design*) exige que la protección se integre desde la arquitectura.

Dos estrategias principales:

| Estrategia | Ejemplo | Ventajas | Inconvenientes |
|------------|---------|----------|----------------|
| **Proyectos separados** | `pa-dev.bronze_personio` / `pa-prod.bronze_personio` | Aislamiento total, IAM independiente | Duplicación de configuración |
| **Datasets con sufijo** | `bronze_personio_dev` / `bronze_personio_prod` | Simplicidad, un solo proyecto | Sin aislamiento de IAM por proyecto |

Para este curso usamos datasets con sufijo. En producción real con datos de empleados, **proyectos separados** es la recomendación.

In [17]:
# --- Crear datasets dev/staging como ejemplo de separación de entornos ---
DEV_DATASETS = {
    "bronze_personio_dev": "DEV — capa Bronze para desarrollo y experimentación",
    "silver_personio_dev": "DEV — capa Silver para testing de transformaciones",
    "gold_people_analytics_dev": "DEV — capa Gold para validación de métricas",
}

print("CREANDO ENTORNO DE DESARROLLO (datasets _dev)")
print("=" * 70)

for dataset_id, description in DEV_DATASETS.items():
    dataset_ref = f"{PROJECT_ID}.{dataset_id}"
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = REGION
    dataset.description = description
    dataset.labels = {"environment": "dev", "team": "people-analytics", "data_classification": "anonymized"}
    try:
        bq_client.create_dataset(dataset)
        print(f"  Creado: {dataset_id}")
    except Conflict:
        print(f"  Existe: {dataset_id}")

# --- Copiar un subconjunto de datos a dev (para testing) ---
print(f"\nCargando datos de prueba en dev (subconjunto de prod)...")

sql_copy_to_dev = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.silver_personio_dev.dim_employee` AS
SELECT * FROM `{PROJECT_ID}.silver_personio.dim_employee`
LIMIT 50  -- Solo 50 empleados en dev (minimización de datos)
"""
bq_client.query(sql_copy_to_dev).result()
t = bq_client.get_table(f"{PROJECT_ID}.silver_personio_dev.dim_employee")
print(f"  silver_personio_dev.dim_employee: {t.num_rows} filas (prod tiene 209)")
print(f"\n  En dev solo se usan datos anonimizados y subconjuntos reducidos.")
print(f"  Regla de oro: datos reales de producción NUNCA se copian sin anonimizar.")

CREANDO ENTORNO DE DESARROLLO (datasets _dev)
  Existe: bronze_personio_dev
  Existe: silver_personio_dev
  Existe: gold_people_analytics_dev

Cargando datos de prueba en dev (subconjunto de prod)...
  silver_personio_dev.dim_employee: 50 filas (prod tiene 209)

  En dev solo se usan datos anonimizados y subconjuntos reducidos.
  Regla de oro: datos reales de producción NUNCA se copian sin anonimizar.


In [18]:
# --- Labels: metadata para organización y FinOps ---
print("LABELS DE DATASETS (para FinOps y gobernanza)")
print("=" * 70)

# Añadir labels a los datasets de producción
prod_labels = {
    "environment": "prod",
    "team": "people-analytics",
    "data_classification": "confidential",
    "cost_center": "pa-001",
}

for dataset_id in ["bronze_personio", "silver_personio", "gold_people_analytics", "ml_features"]:
    dataset_ref = bq_client.get_dataset(f"{PROJECT_ID}.{dataset_id}")
    dataset_ref.labels = prod_labels
    bq_client.update_dataset(dataset_ref, ["labels"])
    print(f"  {dataset_id}: {dict(dataset_ref.labels)}")

print(f"\nLos labels permiten:")
print(f"  - Filtrar costes por 'cost_center' en Cloud Billing")
print(f"  - Identificar datasets por 'environment' (dev/staging/prod)")
print(f"  - Clasificar datos por 'data_classification' (public/internal/confidential)")
print(f"  - Asignar responsabilidad por 'team'")

LABELS DE DATASETS (para FinOps y gobernanza)
  bronze_personio: {'environment': 'prod', 'team': 'people-analytics', 'data_classification': 'confidential', 'cost_center': 'pa-001'}
  silver_personio: {'environment': 'prod', 'team': 'people-analytics', 'data_classification': 'confidential', 'cost_center': 'pa-001'}
  gold_people_analytics: {'environment': 'prod', 'team': 'people-analytics', 'data_classification': 'confidential', 'cost_center': 'pa-001'}
  ml_features: {'environment': 'prod', 'team': 'people-analytics', 'data_classification': 'confidential', 'cost_center': 'pa-001'}

Los labels permiten:
  - Filtrar costes por 'cost_center' en Cloud Billing
  - Identificar datasets por 'environment' (dev/staging/prod)
  - Clasificar datos por 'data_classification' (public/internal/confidential)
  - Asignar responsabilidad por 'team'


---
## 9. Resumen de la arquitectura completa

En esta sesión (Módulos 1 + 2) hemos construido una arquitectura profesional de datos para People Analytics en GCP:

In [19]:
# --- Inventario final completo ---
print("ARQUITECTURA COMPLETA — SESIÓN 1")
print("=" * 80)

# BigQuery
print("\n📦 BIGQUERY (Data Warehouse)")
print("-" * 80)
all_datasets = list(bq_client.list_datasets())
for ds in all_datasets:
    dataset_ref = bq_client.get_dataset(ds.reference)
    tables = list(bq_client.list_tables(ds.reference))
    env = dataset_ref.labels.get("environment", "?")
    classification = dataset_ref.labels.get("data_classification", "?")
    print(f"\n  {ds.dataset_id} [env={env}, class={classification}]")
    for t in tables:
        table_info = bq_client.get_table(t.reference)
        ttype = "VIEW" if table_info.table_type == "VIEW" else "EXTERNAL" if table_info.table_type == "EXTERNAL" else "TABLE"
        print(f"    {ttype:<8} {t.table_id}: {table_info.num_rows or '?'} filas")

# Cloud Storage
print(f"\n📁 CLOUD STORAGE (Data Lake)")
print("-" * 80)
print(f"  Bucket: gs://{BUCKET_NAME}/")
blobs = list(bucket.list_blobs())
total_size = sum(b.size for b in blobs)
print(f"  Archivos: {len(blobs)}")
print(f"  Tamaño total: {total_size / 1024:.0f} KB")

prefixes = set()
for b in blobs:
    parts = b.name.split("/")
    if len(parts) >= 2:
        prefixes.add(f"  {parts[0]}/{parts[1]}/")
for p in sorted(prefixes):
    print(f"    {p}")

# Analytics Hub
print(f"\n🔗 ANALYTICS HUB")
print("-" * 80)
print(f"  Exchange: {EXCHANGE_ID}")
print(f"  Listing: gold_people_analytics → {LISTING_ID}")

# Resumen de arquitectura
print(f"\n{'=' * 80}")
print("DIAGRAMA DE ARQUITECTURA")
print("=" * 80)
print("""
  FUENTES                    INGESTA                  ALMACENAMIENTO
  ┌──────────┐              ┌───────────┐            ┌─────────────────────────────────┐
  │ Personio │──── API ────►│ Cloud     │──── CSV ──►│ GCS: bronze/personio/raw/       │
  │ (HRIS)   │              │ Functions │            │      (landing zone)              │
  ├──────────┤              │ / manual  │            ├─────────────────────────────────┤
  │ Nómina   │──── CSV ────►│           │── Parquet─►│ GCS: silver/personio/parquet/   │
  │ (payroll)│              └───────────┘            │      (backup, BigLake)           │
  └──────────┘                    │                  └──────────┬──────────────────────┘
                                  │                             │
                                  ▼                             ▼
                         ┌──────────────────────────────────────────────────┐
                         │              BIGQUERY                             │
                         │  ┌──────────┐  ┌──────────┐  ┌──────────┐       │
                         │  │ BRONZE   │→ │ SILVER   │→ │  GOLD    │       │
                         │  │ raw_*    │  │ dim_*    │  │ headcount│       │
                         │  │ ext_*    │  │ fact_*   │  │ comp.    │       │
                         │  │ (BigLake)│  │          │  │ turnover │       │
                         │  └──────────┘  └──────────┘  └────┬─────┘       │
                         │                                    │             │
                         │  ┌──────────┐                      │             │
                         │  │ml_features│ ←───────────────────┘             │
                         │  └──────────┘                                    │
                         └──────────┬────────────────────┬─────────────────┘
                                    │                    │
                    ┌───────────────┤                    ├──────────────────┐
                    ▼               ▼                    ▼                  ▼
              ┌──────────┐  ┌────────────┐      ┌────────────┐    ┌──────────────┐
              │ Vertex AI│  │ Looker     │      │ Analytics  │    │ Connected    │
              │ BQML     │  │ Studio     │      │ Hub        │    │ Sheets       │
              │ AutoML   │  │ Dashboards │      │ (sharing)  │    │ (RRHH)       │
              └──────────┘  └────────────┘      └────────────┘    └──────────────┘

  SEGURIDAD: IAM (mínimo privilegio) + Labels (FinOps) + Vistas autorizadas
  ENTORNOS: prod (datos reales) / dev (subconjuntos anonimizados)
  REGION:   europe-southwest1 (GDPR)
""")

ARQUITECTURA COMPLETA — SESIÓN 1

📦 BIGQUERY (Data Warehouse)
--------------------------------------------------------------------------------

  bronze_personio [env=prod, class=confidential]
    EXTERNAL ext_employee_parquet: ? filas
    TABLE    raw_employee_data: 209 filas
    TABLE    raw_employee_history: 1189 filas
    TABLE    raw_gross_salary: 994 filas

  bronze_personio_dev [env=dev, class=anonymized]

  gold_people_analytics [env=prod, class=confidential]
    TABLE    compensation_analysis: 121 filas
    TABLE    headcount_monthly: 317 filas
    TABLE    turnover_metrics: 209 filas
    VIEW     v_headcount_resumen: ? filas

  gold_people_analytics_dev [env=dev, class=anonymized]

  ml_features [env=prod, class=confidential]

  silver_personio [env=prod, class=confidential]
    TABLE    dim_employee: 209 filas
    TABLE    fact_payroll_monthly: 994 filas
    TABLE    fact_salary_history: 1189 filas

  silver_personio_dev [env=dev, class=anonymized]
    TABLE    dim_employee:

---
## 10. Siguientes pasos

### Lo que hemos construido en la Sesión 1 (Módulos 1 + 2):

**Módulo 1 — Contexto y datos:**
1. Entorno híbrido (Vertex AI Workbench / local)
2. Datasets Medallion en BigQuery (Bronze → Silver → Gold → ml_features)
3. Ingesta de datos reales anonimizados de Personio
4. Transformaciones Silver: `dim_employee`, `fact_salary_history`, `fact_payroll_monthly`
5. Métricas Gold: headcount, compensación, rotación
6. Análisis PA: brecha salarial, compa-ratio, k-anonimidad, proxy discrimination

**Módulo 2 — Arquitectura GCP:**
1. Exploración con `INFORMATION_SCHEMA` (esquemas, particiones, almacenamiento)
2. Cloud Storage como Data Lake (bucket con estructura medallion, lifecycle)
3. BigLake: tablas externas sobre Parquet en GCS
4. Auditoría de jobs y control de costes (INFORMATION_SCHEMA.JOBS)
5. IAM: permisos por dataset, vistas autorizadas, política de acceso por capa
6. Analytics Hub: exchange y listing para compartición segura
7. Separación dev/test/prod: datasets con labels, subconjuntos anonimizados

### En las próximas sesiones:
- **Sesión 2**: Ingesta con Pub/Sub + pipelines event-driven (Eventarc)
- **Sesión 3**: Orquestación (Workflows vs Composer) + BigQuery avanzado (Stored Procs, UDFs)
- **Sesión 4**: SQL avanzado sobre estos datos + Dataform como capa de transformación
- **Sesión 9**: Vertex AI — modelo de predicción de rotación sobre `ml_features`